# Dummy HAB classifier

This notebook establishes a majority-class baseline using `thetao_mean` as the only feature. The labeled file contains observed HAB events and synthetic non-events, so it can be evaluated with accuracy. The Copernicus physics file is unlabeled, so predictions on it are only a pipeline smoke test.

In [1]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [2]:
features = pd.read_csv('../data/processed/hab_events_with_environmental_features.csv')

X = features[['thetao_mean']]
y = features['is_synthetic'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

print(f'Training rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')
print(f'Class counts:\n{y.value_counts().sort_index()}')
print(f'Predicted class: {baseline.classes_[baseline.class_prior_.argmax()]}')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(f'Balanced accuracy: {balanced_accuracy_score(y_test, y_pred):.3f}')
print(classification_report(
    y_test,
    y_pred,
    target_names=['HAB event', 'Synthetic non-event'],
    zero_division=0,
))

Training rows: 155
Test rows: 39
Class counts:
is_synthetic
0    97
1    97
Name: count, dtype: int64
Predicted class: 1
Accuracy: 0.487
Balanced accuracy: 0.500
                     precision    recall  f1-score   support

          HAB event       0.00      0.00      0.00        20
Synthetic non-event       0.49      1.00      0.66        19

           accuracy                           0.49        39
          macro avg       0.24      0.50      0.33        39
       weighted avg       0.24      0.49      0.32        39



## Smoke test on unlabeled Copernicus features

These rows do not have `is_synthetic`, so accuracy cannot be calculated. Since this is a dummy classifier, every row receives the same majority-class prediction.

In [3]:
copernicus = pd.read_csv('../data/processed/copernicus_physics_features.csv')
copernicus['predicted_is_synthetic'] = baseline.predict(copernicus[['thetao_mean']])
copernicus['predicted_hab'] = copernicus['predicted_is_synthetic'].eq(0)

print('This is a pipeline check, not an accuracy evaluation.')
display(copernicus[['haedat_row_id', 'thetao_mean', 'predicted_hab']].head())

This is a pipeline check, not an accuracy evaluation.


,haedat_row_id,thetao_mean,predicted_hab
0,0,7.744200,False
1,1,6.860889,False
2,2,6.631021,False
3,3,7.460264,False
4,4,5.197024,False
